# Phase 1: Baseline Text Classification
## Interactive Tutorial

This notebook guides you through Phase 1 of the content moderation system:
1. Dataset exploration
2. Model training
3. Confidence calibration
4. Threshold optimization

**Prerequisites**: Run `pip install -r ../requirements.txt`

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Dataset Exploration

In [ ]:
# Download dataset (if not already downloaded)
from download_data import download_jigsaw_dataset

data_dir = Path('./data')
data_dir.mkdir(exist_ok=True)

# This will download from HuggingFace
dataset = download_jigsaw_dataset(str(data_dir))

In [ ]:
# Load the training data
train_df = pd.read_csv(data_dir / 'train.csv')

print(f"Dataset shape: {train_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")
print(f"\nFirst few rows:")
train_df.head()

In [ ]:
# Label distribution
label_cols = ['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Bar plot of label counts
label_counts = train_df[label_cols].sum().sort_values(ascending=False)
axes[0].bar(range(len(label_counts)), label_counts.values)
axes[0].set_xticks(range(len(label_counts)))
axes[0].set_xticklabels(label_counts.index, rotation=45)
axes[0].set_ylabel('Count')
axes[0].set_title('Label Distribution')

# Add percentages
for i, v in enumerate(label_counts.values):
    percentage = (v / len(train_df)) * 100
    axes[0].text(i, v + 500, f'{percentage:.2f}%', ha='center')

# Number of labels per comment
num_labels = train_df[label_cols].sum(axis=1)
axes[1].hist(num_labels, bins=range(8), edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Number of Labels')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Number of Labels per Comment')
axes[1].set_xticks(range(7))

plt.tight_layout()
plt.show()

print(f"\nClean comments (no labels): {(num_labels == 0).sum()} ({(num_labels == 0).sum()/len(train_df)*100:.2f}%)")
print(f"Toxic comments (>=1 label): {(num_labels > 0).sum()} ({(num_labels > 0).sum()/len(train_df)*100:.2f}%)")

In [ ]:
# Text length analysis
train_df['text_length'] = train_df['comment_text'].str.len()
train_df['word_count'] = train_df['comment_text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Character length
axes[0].hist(train_df['text_length'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(train_df['text_length'].median(), color='red', linestyle='--', label=f'Median: {train_df["text_length"].median():.0f}')
axes[0].set_xlabel('Text Length (characters)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Text Length')
axes[0].legend()

# Word count
axes[1].hist(train_df['word_count'], bins=50, edgecolor='black', alpha=0.7)
axes[1].axvline(train_df['word_count'].median(), color='red', linestyle='--', label=f'Median: {train_df["word_count"].median():.0f}')
axes[1].set_xlabel('Word Count')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Word Count')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nText length statistics:")
print(train_df['text_length'].describe())

In [ ]:
# Sample comments
print("=" * 80)
print("SAMPLE TOXIC COMMENTS")
print("=" * 80)

for label in label_cols:
    print(f"\n### {label.upper()} ###")
    samples = train_df[train_df[label] == 1].sample(2, random_state=42)
    for idx, row in samples.iterrows():
        print(f"- {row['comment_text'][:200]}...")

print("\n" + "=" * 80)
print("SAMPLE CLEAN COMMENTS")
print("=" * 80)
clean_samples = train_df[train_df[label_cols].sum(axis=1) == 0].sample(5, random_state=42)
for idx, row in clean_samples.iterrows():
    print(f"- {row['comment_text'][:200]}...")

## 2. Model Training (Step 1.1)

For training, we'll use the command-line script. However, here's what happens under the hood:

In [ ]:
# To train, run in terminal:
# python train_classifier.py --config configs/baseline.yaml

# Or train on sample data for quick test:
import yaml

config_path = 'configs/baseline.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Training Configuration:")
print(f"Model: {config['model']['name']}")
print(f"Batch size: {config['training']['batch_size']}")
print(f"Learning rate: {config['training']['learning_rate']}")
print(f"Epochs: {config['training']['num_epochs']}")
print(f"\nTo start training, run:")
print("python train_classifier.py --config configs/baseline.yaml")

## 3. Load Trained Model Results

After training completes, we can analyze the results:

In [ ]:
# Check if model exists
model_path = Path('models/best_model.pt')

if model_path.exists():
    checkpoint = torch.load(model_path, map_location='cpu')
    metrics = checkpoint['metrics']
    
    print("Model Training Results:")
    print(f"Epoch: {checkpoint['epoch']}")
    print(f"\nOverall Metrics:")
    print(f"  F1:        {metrics['overall_f1']:.4f}")
    print(f"  Precision: {metrics['overall_precision']:.4f}")
    print(f"  Recall:    {metrics['overall_recall']:.4f}")
    print(f"  AUC:       {metrics['overall_auc']:.4f}")
    
    print(f"\nPer-Label Metrics:")
    for label in label_cols:
        print(f"  {label:15s} - F1: {metrics[f'{label}_f1']:.4f}, AUC: {metrics[f'{label}_auc']:.4f}")
else:
    print("Model not found. Please run training first:")
    print("python train_classifier.py --config configs/baseline.yaml")

In [ ]:
# Load validation predictions for analysis
pred_path = Path('models/val_predictions.npy')
label_path = Path('models/val_labels.npy')

if pred_path.exists() and label_path.exists():
    predictions = np.load(pred_path)
    labels = np.load(label_path)
    
    print(f"Predictions shape: {predictions.shape}")
    print(f"Labels shape: {labels.shape}")
    
    # Plot prediction distributions
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    
    for i, label in enumerate(label_cols):
        # Positive class predictions
        pos_preds = predictions[labels[:, i] == 1, i]
        neg_preds = predictions[labels[:, i] == 0, i]
        
        axes[i].hist(neg_preds, bins=50, alpha=0.6, label='Negative', color='green')
        axes[i].hist(pos_preds, bins=50, alpha=0.6, label='Positive', color='red')
        axes[i].set_xlabel('Predicted Probability')
        axes[i].set_ylabel('Count')
        axes[i].set_title(f'{label} - Prediction Distribution')
        axes[i].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("Predictions not found. Please run training first.")

## 4. Calibration Analysis (Step 1.2)

After running calibration script, we can visualize the results:

In [ ]:
# To run calibration:
# python calibrate_thresholds.py --config configs/baseline.yaml

import json

calib_path = Path('models/calibration/calibration_results.json')

if calib_path.exists():
    with open(calib_path, 'r') as f:
        calib_results = json.load(f)
    
    print("Calibration Results:")
    print(f"Method: {calib_results['method']}\n")
    
    for label, results in calib_results['labels'].items():
        print(f"\n{'='*60}")
        print(f"{label.upper()}")
        print(f"{'='*60}")
        
        print(f"\nCalibration Improvement:")
        print(f"  ECE Before: {results['ece_before']:.4f}")
        print(f"  ECE After:  {results['ece_after']:.4f}")
        print(f"  Improvement: {(results['ece_before'] - results['ece_after']):.4f}")
        
        print(f"\nOptimal Threshold: {results['optimal_threshold']:.4f}")
        print(f"  Precision: {results['threshold_metrics']['precision']:.4f}")
        print(f"  Recall:    {results['threshold_metrics']['recall']:.4f}")
        print(f"  F1:        {results['threshold_metrics']['f1']:.4f}")
        
        tiers = results['tier_thresholds']
        print(f"\nEnforcement Tiers:")
        print(f"  Auto-remove (T > {tiers['t_high']:.3f}):")
        print(f"    - Volume: {tiers['auto_remove']['percentage']:.2f}%")
        print(f"    - Precision: {tiers['auto_remove']['precision']:.4f}")
        
        print(f"  Human-review ({tiers['t_low']:.3f} < T < {tiers['t_high']:.3f}):")
        print(f"    - Volume: {tiers['human_review']['percentage']:.2f}%")
        print(f"    - Positive rate: {tiers['human_review']['positive_rate']:.4f}")
        
        print(f"  Auto-approve (T < {tiers['t_low']:.3f}):")
        print(f"    - Volume: {tiers['auto_approve']['percentage']:.2f}%")
        print(f"    - FN rate: {tiers['auto_approve']['false_negative_rate']:.4f}")
else:
    print("Calibration results not found. Please run:")
    print("python calibrate_thresholds.py --config configs/baseline.yaml")

## 5. Interactive Inference

Test the model on your own text:

In [ ]:
from transformers import AutoTokenizer
from train_classifier import ToxicCommentClassifier

def predict_toxicity(text, model, tokenizer, config, calibration):
    """Predict toxicity for a given text."""
    # Tokenize
    encoding = tokenizer(
        text,
        max_length=config['model']['max_length'],
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(
            input_ids=encoding['input_ids'],
            attention_mask=encoding['attention_mask']
        )
        probs = torch.sigmoid(outputs['logits']).numpy()[0]
    
    # Format results
    results = []
    for i, label in enumerate(config['labels']['names']):
        prob = probs[i]
        if label in calibration['labels']:
            thresh = calibration['labels'][label]['optimal_threshold']
            tiers = calibration['labels'][label]['tier_thresholds']
            
            if prob >= tiers['t_high']:
                action = '🚫 AUTO-REMOVE'
            elif prob >= tiers['t_low']:
                action = '👁️ HUMAN REVIEW'
            else:
                action = '✅ AUTO-APPROVE'
        else:
            action = 'N/A'
        
        results.append({
            'label': label,
            'probability': prob,
            'action': action
        })
    
    return results

# Load model if available
if model_path.exists() and calib_path.exists():
    # Load model
    checkpoint = torch.load(model_path, map_location='cpu')
    config = checkpoint['config']
    
    model = ToxicCommentClassifier(
        model_name=config['model']['name'],
        num_labels=config['model']['num_labels'],
        dropout=config['model']['dropout'],
        hidden_size=config['model']['hidden_size'],
        use_intermediate_layer=config['model']['use_intermediate_layer']
    )
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(config['model']['name'])
    
    # Load calibration
    with open(calib_path, 'r') as f:
        calibration = json.load(f)
    
    print("Model loaded successfully!\n")
    
    # Test examples
    test_texts = [
        "This is a great article, thank you for sharing!",
        "You are an idiot and I hate you!",
        "I disagree with your opinion, but respect your view.",
        "Go kill yourself, nobody likes you."
    ]
    
    for text in test_texts:
        print("=" * 80)
        print(f"TEXT: {text}")
        print("=" * 80)
        
        results = predict_toxicity(text, model, tokenizer, config, calibration)
        
        for r in results:
            print(f"{r['label']:15s}: {r['probability']:.4f} - {r['action']}")
        print()
    
    # Interactive prediction
    print("\n" + "="*80)
    print("Try your own text! (Enter empty line to skip)")
    print("="*80)
    
    user_text = input("Enter text: ")
    if user_text.strip():
        results = predict_toxicity(user_text, model, tokenizer, config, calibration)
        print(f"\nResults for: {user_text}")
        for r in results:
            print(f"{r['label']:15s}: {r['probability']:.4f} - {r['action']}")
else:
    print("Model not available. Please complete training and calibration first.")

## Summary

✅ **Phase 1 Complete!**

You now have:
1. A fine-tuned toxic comment classifier
2. Calibrated confidence scores
3. Optimal classification thresholds
4. Three-tier enforcement system

**Next Steps**: Move to Phase 2 for multilingual support!